In [ ]:
import re
import pandas as pd
import numpy as np

years_annual = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]  #cn.years_annual

##### Functions to carry over to geospatial implementation

In [ ]:
""" Regex-based land-use reclassification rules
These rules replace the default land use classes.
1. Convert annual GLAD LC values to LU tokens.
2. Use regex to identify token patterns for exceptions.
3. Reclassify token arrays and assign a matching node_code array.

Node codes used here:
1) Settlements and Infrastructure:
    10 = Built from GLAD data
    11 = Built following tall veg loss before built LC


2) Cropland:
    20 = Crop from GLAD data
    21 = Crop from oil palm extent or planting year
    22 = Crop from SDPT tree crop extent
    23 = Crop from permanent agriculture driver
    24 = Crop following tall veg loss before crop LC
    25 = Crop from majority years in mixed LC prior to built LC


3) Forest:
    30  = Forest from GLAD tall vegetation
    31  = Forest from SDPT planted forest extent
    32  = Forest from GMW mangrove extent
    33X = Unstocked forest from TCL + drivers rules
        333 = Forest from shifting cultivation driver
        334 = Forest from logging driver
        335 = Forest from wildfire driver
        337 = Forest from natural disturbance driver
    34 = Unstocked forest after TCL and before oil palm planting
    35 = Forest from vegetation/bare to built transition rule
    36 = Forest from vegetation/bare to crop transition rule
    37 = Forest from mixed tall/short vegetation rule
    38 = Forest from mixed vegetation/water rule
    39 = Forest from majority years in mixed class rule


4) Grassland:
    40 = Grass from GLAD short vegetation
    41 = Grass from TCL + permanent agriculture driver + GPW cultivated grassland extent (assume rangeland)
    42x = Grass from TCL + driver rule
        420 = Grass from unknown driver
        422 = Grass from hard commodities driver
        426 = Grass from settlements/infrastructure driver
    43 = Grass from vegetation/bare to built transition rule
    44 = Grass from vegetation/bare to crop transition rule
    45 = Grass from mixed tall/short vegetation rule
    46 = Grass from mixed vegetation/water rule
    47 = Grass from majority years in mixed class rule


5) Wetland:
    50 = Wetland from GLAD data
    51 = Wetland from vegetation/water transition rule
    52 = Wetland from majority years in mixed water rule


6) Other
    60 = Bare from GLAD data
    61 = Bare from majority years in mixed bare/grass rule

    70 = Water from GLAD data
    71 = Water from vegetation/water transition rule
    72 = Water from majority years in mixed water rule

    80 = Snow/ice from GLAD data
    81 = Ice from majority years in mixed class rule
"""

# IPCC Land use hierarchy: Settlements > Cropland > Forest Land > Grassland > Wetlands > Other
# Default GLAD LC numeric values
settlement_lc   = {250}                                         # Built up
cropland_lc     = {244}                                         # Cropland
forest_lc       = set(range(27, 49)) | set(range(127, 149))     # Tall vegetation
grass_lc        = set(range(5, 27)) | set(range(105, 127))      # Short veg
wetland_lc      = set(range(200, 205))                          # Wetland
bare_lc         = set(range(0, 5)) | set(range(100, 105))       # Bare
water_lc        = set(range(205, 208)) | {254}                  # Open water
ice_lc          = {241}                                         # Snow/ice

# Lookup table to go from GLAD LC code -> default LU token
lc_token_map = {
    **{v: "S" for v in settlement_lc},
    **{v: "C" for v in cropland_lc},
    **{v: "F" for v in forest_lc},
    **{v: "G" for v in grass_lc},
    **{v: "W" for v in wetland_lc},
    **{v: "B" for v in bare_lc},
    **{v: "O" for v in water_lc},
    **{v: "I" for v in ice_lc},
}

# Function to get land use token per land cover numeric value (tokens used for regex exception rules)
def token_for_lc(v):
    if v not in lc_token_map:
        raise ValueError(f"Unknown GLCLU code: {v}")
    return lc_token_map[v]

# Node code values based on what exception was applied
node_code_map = {
    "built_glad": 10,
    "built_tall_veg_loss": 11,

    "crop_glad": 20,
    "crop_oil_palm": 21,
    "crop_sdpt_tree_crop": 22,
    "crop_perm_ag_driver": 23,
    "crop_tall_veg_loss": 24,
    "crop_glad_majority_years": 25,

    "forest_glad": 30,
    "forest_sdpt_planted_forest": 31,
    "forest_gmw_mangrove": 32,
    "forest_shift_cult_driver": 333,
    "forest_logging_driver": 334,
    "forest_wildfire_driver": 335,
    "forest_nat_dist_driver": 337,
    "forest_unstocked_pre_oil_palm": 34,
    "forest_veg_bare_built_mix": 35,
    "forest_veg_bare_crop_mix": 36,
    "forest_tall_short_mix": 37,
    "forest_veg_water_mix": 38,
    "forest_glad_majority_years": 39,

    "grass_glad": 40,
    "grass_gpw": 41,
    "grass_hard_commod_driver": 422,
    "grass_settlement_driver": 426,
    "grass_unknown_driver": 420,
    "grass_veg_bare_built_mix": 43,
    "grass_veg_bare_crop_mix": 44,
    "grass_tall_short_mix": 45,
    "grass_veg_water_mix": 46,
    "grass_glad_majority_years": 47,


    "wetland_glad": 50,
    "wetland_veg_water_mix": 51,
    "wetland_glad_majority_years": 52,

    "bare_glad": 60,
    "bare_glad_majority_years": 61,

    "water_glad": 70,
    "water_veg_water_mix": 71,
    "water_glad_majority_years": 72,

    "ice_glad": 80,
    "ice_glad_majority_years": 81,
}

# Default node codes before rules are applied
def default_node_code(token):
    if token == "S":
        return node_code_map["built_glad"]
    if token == "C":
        return node_code_map["crop_glad"]
    if token == "F":
        return node_code_map["forest_glad"]
    if token == "G":
        return node_code_map["grass_glad"]
    if token == "W":
        return node_code_map["wetland_glad"]
    if token == "B":
        return node_code_map["bare_glad"]
    if token == "O":
        return node_code_map["water_glad"]
    if token == "I":
        return node_code_map["ice_glad"]
    return None

# Function to override default values based on regex rules
def set_tokens(tokens, node_codes, indices, new_token, node_code, initial_tokens=None):
    for i in indices:
        old_token = initial_tokens[i] if initial_tokens is not None else tokens[i]
        tokens[i] = new_token

        # Only overwrite node code if LU token changed from original LC token
        if old_token != new_token:
            node_codes[i] = node_code

def apply_tokens(lu_dict, indices, new_token, node_code):
    set_tokens(lu_dict["tokens"], lu_dict["node_codes"], indices, new_token, node_code, lu_dict["initial_tokens"])

# Select pre-transition token by count. Ties go to the earlier token in priority_order.
def majority_token(tokens, candidates, priority_order):
    present = [t for t in candidates if t in tokens]
    return max(present, key=lambda t: (tokens.count(t), -priority_order.index(t)))

# Converts char tokens to final int values in LU map
lu_token_map = {
    "S": 1,
    "C": 2,
    "F": 3,
    "G": 4,
    "W": 5,
    "B": 6,
    "O": 7,
    "I": 8,
}

In [ ]:
# Checks if an oil palm planting year related transition happens in the timeseries.
def has_planting_transition(lu_dict):
    planting_year = lu_dict.get("planting_year", 0)
    return (planting_year > min(years_annual) and planting_year <= max(years_annual))

# Gets oil palm planting year index in timeseries.
def planting_idx(planting_year):
    if planting_year <= min(years_annual):
        return 0
    if planting_year > max(years_annual):
        return None
    return int(planting_year - min(years_annual))

# Checks if there was TCL up to 5 years before oil palm planting year. If so, considered F->C transition.
def tcl_prior_to_planting(tcl_year, planting_year):
    n_years = 5     # number of years between TCL and oil palm planting year allowed to be considered F -> C conversion
    return (tcl_year > 0 and planting_year > 0 and planting_year - n_years <= tcl_year < planting_year)

In [ ]:
# Checks that there is only one land use transition during the entire timeseries. If not, prints pixel information.
def check_single_lu_transition(lu_dict, lu_ts):
    transition_count = sum(lu_ts[i] != lu_ts[i - 1] for i in range(1, len(lu_ts)))

    if transition_count > 1:
        debug_info = {
            k: v for k, v in lu_dict.items()
            if k not in {"node_codes"}
        }

        print(
            f"\nWARNING: More than one LU transition detected.\n"
            f"transition_count: {transition_count}\n"
            f"lu_ts: {lu_ts}\n"
            f"debug_info: {debug_info}\n"
        )

In [ ]:
def apply_extent_rules(lu_dict):
    tokens = lu_dict["tokens"]

    crop_reclass_idx = [i for i, token in enumerate(tokens) if token in {"F", "G", "W", "B", "O", "I"}]
    forest_reclass_idx = [i for i, token in enumerate(tokens) if token in {"G", "W", "B", "O", "I"}]
    # TODO: May want to consider not including wetland? Include O?

    # Get oil palm planting year
    crop_extent = lu_dict["sdpt_tree_crop"] or lu_dict["sdpt_oil_palm"]
    planting_year = lu_dict.get("planting_year", 0)
    planting_later = planting_year > min(years_annual)

    # Crop is highest priority and extents are applied in this order: pre-2000 oil palm plantation -> Descals oil palm -> SDPT tree crop
    if lu_dict["pre_2000_plantation"]:
        apply_tokens(lu_dict, crop_reclass_idx, "C", node_code_map["crop_oil_palm"])
        return True
    if crop_extent and planting_later:
        return False    # Don't apply oil palm exception in SDPT extent if planting year hasn't happened yet
    if crop_extent:
        node_code = node_code_map["crop_oil_palm"] if lu_dict["sdpt_oil_palm"] else node_code_map["crop_sdpt_tree_crop"]
        apply_tokens(lu_dict, crop_reclass_idx, "C", node_code)
        return True

    # If no crop extent applies, forest extents are applied by this order: GMW mangrove -> SDPT planted forest
    if lu_dict["gmw_mangrove"]:
        apply_tokens(lu_dict, forest_reclass_idx, "F", node_code_map["forest_gmw_mangrove"])
        return True
    if lu_dict["sdpt_planted_forest"]:
        apply_tokens(lu_dict, forest_reclass_idx, "F", node_code_map["forest_sdpt_planted_forest"])
        return True
    return False

In [ ]:
# Mix of 2 or more LC classes -> built
def apply_built_transition(lu_dict):
    tokens = lu_dict["tokens"]
    token_seq = "".join(tokens)

    if "S" not in token_seq:
        return

    # Require at least 2 non-S LC classes
    non_s_classes = set(tokens) - {"S"}
    if len(non_s_classes) < 2:
        return

    pre_node_map = {
        "C": node_code_map["crop_glad_majority_years"],
        "F": node_code_map["forest_glad_majority_years"],
        "G": node_code_map["grass_glad_majority_years"],
        "W": node_code_map["wetland_glad_majority_years"],
        "B": node_code_map["bare_glad_majority_years"],
        "O": node_code_map["water_glad_majority_years"],
        "I": node_code_map["ice_glad_majority_years"],
    }

    first_s_idx = token_seq.find("S")

    # Go down hierarchy. C is handled as tie-breaker when present.
    for candidate in ["F", "G", "W", "B", "O", "I"]:
        if candidate not in token_seq:
            continue

        pre_token = candidate

        # Count majority pre-token. Tie goes to C.
        if "C" in token_seq:
            c_count = tokens.count("C")
            candidate_count = tokens.count(candidate)
            if c_count >= candidate_count:
                pre_token = "C"

        # F uses first F loss; everything else uses first S.
        if candidate == "F" and pre_token == "F":
            transition_match = re.search(r"F+[SCGWBOI]", token_seq)
            if not transition_match:
                transition_idx = first_s_idx
            else:
                transition_idx = transition_match.end() - 1
        else:
            transition_idx = first_s_idx

        apply_tokens(lu_dict, range(0, transition_idx), pre_token, pre_node_map[pre_token])
        apply_tokens(lu_dict, range(transition_idx, len(tokens)), "S", node_code_map["built_tall_veg_loss"])

        return

#TODO: Use TCL up to 5 years prior for F->S exception?

In [ ]:
# Mix of 2 or more LC classes -> crop
def apply_crop_transition(lu_dict):
    tokens = lu_dict["tokens"]
    token_seq = "".join(tokens)

    if "C" not in token_seq:
        return

    # Require at least 2 non-C LC classes
    non_c_classes = set(tokens) - {"C"}
    if len(non_c_classes) < 2:
        return

    pre_node_map = {
        "F": node_code_map["forest_glad_majority_years"],
        "G": node_code_map["grass_glad_majority_years"],
        "W": node_code_map["wetland_glad_majority_years"],
        "B": node_code_map["bare_glad_majority_years"],
        "O": node_code_map["water_glad_majority_years"],
        "I": node_code_map["ice_glad_majority_years"],
    }

    first_c_idx = token_seq.find("C")

    # Go down hierarchy. F uses first F loss; everything else uses first C.
    for candidate in ["F", "G", "W", "B", "O", "I"]:
        if candidate not in token_seq:
            continue

        pre_token = candidate

        if candidate == "F":
            transition_match = re.search(r"F+[CGWBOI]", token_seq)
            if transition_match:
                transition_idx = transition_match.end() - 1
            else:
                transition_idx = first_c_idx
        else:
            transition_idx = first_c_idx

        apply_tokens(lu_dict, range(0, transition_idx), pre_token, pre_node_map[pre_token])
        apply_tokens(lu_dict, range(transition_idx, len(tokens)), "C", node_code_map["crop_tall_veg_loss"])

        return
#TODO: Use TCL up to 5 years prior for F->C exception?

In [ ]:
# Tall vegetation all years
def apply_all_tall_veg(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    driver = lu_dict["driver"]

    tokens = lu_dict["tokens"]
    all_idx = range(len(tokens))

    # If TCL has occurred by the start of timeseries and the driver is permanent ag, assume tall veg is tree crops
    if tcl_prior and driver == 1:
        apply_tokens(lu_dict, all_idx, "C", node_code_map["crop_perm_ag_driver"])

    # # If oil palm planting year in interval, allows for F -> C transitions assuming establishment of tree crops
    # if has_planting_transition(lu_dict):
    #     idx = planting_idx(lu_dict["planting_year"])
    #     apply_tokens(lu_dict, range(idx, len(tokens)), "C", node_code_map["crop_oil_palm"])
    #     return
    #TODO: Do we want to allow for this kind of transition based on oil palm planting year?


In [ ]:
# Short vegetation all years
def apply_all_short_veg(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    driver = lu_dict["driver"]
    planting_year = lu_dict["planting_year"]

    tokens = lu_dict["tokens"]
    all_idx = range(len(lu_dict["tokens"]))

    driver_to_forest_node = {
        3: node_code_map["forest_shift_cult_driver"],
        4: node_code_map["forest_logging_driver"],
        5: node_code_map["forest_wildfire_driver"],
        7: node_code_map["forest_nat_dist_driver"],
    }

    # If oil palm planting year occurs during interval: If TCL up to 5 years prior, assume F -> C transition. Otherwise, assume G -> C transition.
    if has_planting_transition(lu_dict):
        idx = planting_idx(planting_year)
        if tcl_prior_to_planting(lu_dict["tcl_year"], planting_year):
            pre_token = "F"
            pre_node = node_code_map["forest_unstocked_pre_oil_palm"]
            apply_tokens(lu_dict, range(0, idx), pre_token, pre_node)

        apply_tokens(lu_dict, range(idx, len(tokens)), "C", node_code_map["crop_oil_palm"])
        return

    # If TCL has occurred by the start of the timeseries and the driver is permanent ag and not in cultivated grass extent, assume crop the entire timeseries
    elif tcl_prior and driver == 1:
        if not lu_dict["gpw_cultiv_grass"]:
            apply_tokens(lu_dict, all_idx, "C", node_code_map["crop_perm_ag_driver"])
        else:
            apply_tokens(lu_dict, all_idx, "G", node_code_map["grass_gpw"])

    # If TCL has occurred by the start of the timeseries and the driver is shifting cultivation, logging, wildfire, or other natural disturbances, assume unstocked forest the entire timeseries
    elif tcl_prior and driver in driver_to_forest_node:
        apply_tokens(lu_dict, all_idx, "F", driver_to_forest_node[driver])


In [ ]:
# Mix of short veg and tall veg
def apply_tall_short(lu_dict):
    tcl_prior = lu_dict["tcl_prior"]
    tcl_year = lu_dict["tcl_year"]
    tcl_any = tcl_year > 0
    driver = lu_dict["driver"]
    planting_year = lu_dict["planting_year"]

    tokens = lu_dict["tokens"]
    token_seq = "".join(tokens)
    all_idx = range(len(tokens))

    driver_to_forest_node = {
        3: node_code_map["forest_shift_cult_driver"],
        4: node_code_map["forest_logging_driver"],
        5: node_code_map["forest_wildfire_driver"],
        7: node_code_map["forest_nat_dist_driver"],
    }

    driver_to_grass_node = {
        2: node_code_map["grass_hard_commod_driver"],
        6: node_code_map["grass_settlement_driver"],
    }

    # 1) Check if oil palm planting year occurs during interval (regardless of driver + TCL)
    if has_planting_transition(lu_dict):
        # If oil palm planting year in interval, use the first F -> G transition. Else, use oil palm planting year.
        transition_match = re.search(r"F+G", token_seq)
        if transition_match:
            idx = transition_match.end() - 1
        else:
            idx = planting_idx(planting_year)
        pre_plant_tokens = tokens[:idx]

        # If F present before transition or TCL within 5 years before planting, consider it F -> C
        forest_before_planting = ("F" in pre_plant_tokens or tcl_prior_to_planting(tcl_year, planting_year))
        if forest_before_planting:
            pre_token = "F"
            pre_node = node_code_map["forest_unstocked_pre_oil_palm"]
            apply_tokens(lu_dict, range(0, idx), pre_token, pre_node)

        apply_tokens(lu_dict, range(idx, len(tokens)), "C", node_code_map["crop_oil_palm"])
        return

    # 2) If TCL occurred before the timeseries, use permanent agriculture driver to determine LU for all years.
        # If the driver is permanent ag and not in cultivated grass extent, assume crop. Else, assume grass.
    if tcl_prior and driver == 1:
        if not lu_dict["gpw_cultiv_grass"]:
            apply_tokens(lu_dict, all_idx, "C", node_code_map["crop_perm_ag_driver"])
        else:
            apply_tokens(lu_dict, all_idx, "G", node_code_map["grass_gpw"])
        return

    # 3) If TCL during the timeseries, use permanent agriculture driver and first F -> G transition to determine LU transitions:
        # If the driver is permanent ag and not in cultivated grass extent, assume F -> C transition. Else, assume F -> G transition.
    if tcl_any and not tcl_prior and driver == 1:
        match = re.search(r"F+G", token_seq)
        if match:
            transition_idx = match.end() - 1
            # apply_tokens(lu_dict, range(0, transition_idx), "F", node_code_map["forest_glad"])
            # Note: if not using first F->G transition switch node code to forest_tall_short_mix
            if lu_dict["gpw_cultiv_grass"]:
                apply_tokens(lu_dict, range(transition_idx, len(tokens)), "G", node_code_map["grass_gpw"])
            else:
                apply_tokens(lu_dict, range(transition_idx, len(tokens)), "C", node_code_map["crop_perm_ag_driver"])
            return

    # 4) If TCL in any year and driver is temporary, assume forest all years.
        # Temporary drivers are: shifting cultivation, logging, wildfire, and other natural disturbances
    if tcl_any and driver in driver_to_forest_node:
        apply_tokens(lu_dict, all_idx, "F", driver_to_forest_node[driver])
        return

    # 5) If TCL during timeseries, use hard commodities, settlements/ infrastructure, and unknown driver and an F->G transition where it stays G until the end. There must be at least 3 Fs, and at least 3 consecutive Gs until the end to determine LU transitions:
    if tcl_any and not tcl_prior and driver not in {1, 3, 4, 5, 7}:
        terminal_match = re.search(r"F{3,}G{3,}$", token_seq)

        if terminal_match:
            transition_match = re.search(r"F+G", token_seq)     # Get the first F->G transition

            if transition_match:
                transition_idx = transition_match.end() - 1
                apply_tokens(lu_dict, range(0, transition_idx), "F", node_code_map["forest_tall_short_mix"])

                if driver in driver_to_grass_node:
                    grass_node = driver_to_grass_node[driver]
                else:
                    grass_node = node_code_map["grass_unknown_driver"]
                apply_tokens(lu_dict, range(transition_idx, len(tokens)), "G", grass_node)

                return

    # 6) Otherwise use regex fallback (if no oil palm or TCL + driver, LU can only be forest or grass)
    # If there is not at least 3 years F or 3 years G, not enough evidence for a true F/G transition. Use majority land use instead.
    f_count = tokens.count("F")
    g_count = tokens.count("G")
    if g_count < 3:
        apply_tokens(lu_dict, all_idx, "F", node_code_map["forest_glad_majority_years"])
        return
    elif f_count < 3:
        apply_tokens(lu_dict, all_idx, "G", node_code_map["grass_glad_majority_years"])
        return

    # Otherwise F-> G transition needs a terminal G phase that start with 3 consecutive Gs, allow at most one F, end on G.
    # Option to set total number of G years in terminal phase to >= #.
    # TODO: GGGFGG and GGFGGG allowed but not GGFGG?
    else:
        terminal_match = re.search(r"(?P<g>G{3,}(?:F?G*)?)$", token_seq)

        # If no valid terminal G phase, set all years to F
        if not terminal_match:
            apply_tokens(lu_dict, all_idx, "F", node_code_map["forest_tall_short_mix"])
            return

        # Option to make number of Gs in terminal G phase > 3
        terminal_g_start_idx = terminal_match.start("g")
        terminal_g_count = tokens[terminal_g_start_idx:].count("G")
        if terminal_g_count < 3:
            apply_tokens(lu_dict, all_idx, "F", node_code_map["forest_tall_short_mix"])
            return

        # If there is a valid terminal G phase, look for the first F->G transition and sets that as the transition year since that is when the majority of emissions will occur in the vegetation model.
        transition_match = re.search(r"F+G", token_seq)

        if not transition_match:
            apply_tokens(lu_dict, all_idx, "F", node_code_map["forest_tall_short_mix"])
            return

        transition_idx = transition_match.end() - 1
        # apply_tokens(lu_dict, range(0, transition_idx), "F", node_code_map["forest_glad"])
        # Note: if not using first F->G transition switch node code to forest_tall_short_mix
        apply_tokens(lu_dict, range(transition_idx, len(tokens)), "G", node_code_map["grass_tall_short_mix"])

In [ ]:
# Mix of short veg and bare
def apply_short_bare(lu_dict):
    tokens = lu_dict["tokens"]
    token_seq = "".join(tokens)
    all_idx = range(len(tokens))

    # Only considered a LU transition if initial landcover >= 3 consecutive years and final land cover >= 3 consecutive years and there is only 1 transition (i.e. GGGBBBBBBB OR BBBBGGGGGG)
    if re.fullmatch(r"(G{3,}B{3,}|B{3,}G{3,})", token_seq):
        return

    # Otherwise collapse to majority class across all years
    g_count = tokens.count("G")
    b_count = tokens.count("B")

    # If G and B have the same number of years, assume G
    if g_count >= b_count:
        apply_tokens(lu_dict, all_idx, "G", node_code_map["grass_glad_majority_years"])
    else:
        apply_tokens(lu_dict, all_idx, "B", node_code_map["bare_glad_majority_years"])

In [ ]:
# Mix of vegetation/bare and water/wetland
def apply_veg_bare_water(lu_dict):
    tokens = lu_dict["tokens"]
    token_seq = "".join(tokens)
    all_idx = range(len(tokens))

    veg_tokens = {"F", "G", "B"}
    water_tokens = {"W", "O"}

    veg_count = sum(t in veg_tokens for t in tokens)
    water_count = sum(t in water_tokens for t in tokens)

    f_count = tokens.count("F")
    g_count = tokens.count("G")
    b_count = tokens.count("B")
    w_count = tokens.count("W")
    o_count = tokens.count("O")

    # If there are <3 vegetation/bare years, collapse to majority water/wetland. Tie goes to wetland.
    if veg_count < 3:
        if w_count >= o_count:
            apply_tokens(lu_dict, all_idx, "W", node_code_map["wetland_glad_majority_years"])
        else:
            apply_tokens(lu_dict, all_idx, "O", node_code_map["water_glad_majority_years"])
        return

    # If there are <3 water/wetland years, collapse to majority vegetation/bare class. Tie goes to forest.
    if water_count < 3:
        if f_count >= g_count and f_count >= b_count:
            apply_tokens(lu_dict, all_idx, "F", node_code_map["forest_glad_majority_years"])
        elif g_count >= b_count:
            apply_tokens(lu_dict, all_idx, "G", node_code_map["grass_glad_majority_years"])
        else:
            apply_tokens(lu_dict, all_idx, "B", node_code_map["bare_glad_majority_years"])
        return
    #TODO: May want to consider F even when its not majority?

    # Vegetation -> water/wetland transition:
    # 3+ consecutive vegetation/bare years followed by 3+ consecutive water/wetland years until the end.
    transition_match = re.search(r"(?P<veg>[FGB]{3,})(?P<water>[WO]{3,})$", token_seq)

    if transition_match:
        transition_idx = transition_match.start("water")
        pre_tokens = tokens[:transition_idx]
        final_tokens = tokens[transition_idx:]

        # Prominent vegetation/bare class: if F > 2 forest, elif G > 2 grass, else bare.
        if pre_tokens.count("F") > 2:
            pre_token = "F"
            pre_node = node_code_map["forest_glad_majority_years"]
        elif pre_tokens.count("G") > 2:
            pre_token = "G"
            pre_node = node_code_map["grass_glad_majority_years"]
        else:
            pre_token = "B"
            pre_node = node_code_map["bare_glad_majority_years"]

        # Majority water class: if W > 2, wetland; otherwise water.
        if final_tokens.count("W") > 2:
            final_token = "W"
            final_node = node_code_map["wetland_glad_majority_years"]
        else:
            final_token = "O"
            final_node = node_code_map["water_glad_majority_years"]

        apply_tokens(lu_dict, range(0, transition_idx), pre_token, pre_node)
        apply_tokens(lu_dict, range(transition_idx, len(tokens)), final_token, final_node)
        return

   # If enough evidence of both groups (both groups >=3) but no valid transition, consider it wetland all years.
    apply_tokens(lu_dict, all_idx, "W", node_code_map["wetland_glad_majority_years"])
    return
#TODO: change node_codes to veg_water_mix?

In [ ]:
# Mix of wetland and water only
def apply_wetland_water(lu_dict):
    tokens = lu_dict["tokens"]
    token_seq = "".join(tokens)
    all_idx = range(len(tokens))

    # Only considered a LU transition if initial landcover >= 3 consecutive years and final land cover >= 3 consecutive years and there is only 1 transition (i.e. WWWOOOOOOO OR OOOOWWWWWW)
    if re.fullmatch(r"(W{3,}O{3,}|O{3,}W{3,})", token_seq):
        return

    # Otherwise collapse to majority class across all years
    w_count = tokens.count("W")
    o_count = tokens.count("O")

    # If W and O have the same number of years, assume W
    if  w_count >= o_count:
        apply_tokens(lu_dict, all_idx, "W", node_code_map["wetland_glad_majority_years"])
    else:
        apply_tokens(lu_dict, all_idx, "O", node_code_map["water_glad_majority_years"])

In [ ]:
def apply_regex_rules(lc_timeseries, driver, tcl_year, pre_2000_plantation, planting_year, sdpt_oil_palm, sdpt_tree_crop, sdpt_planted_forest, gmw_mangrove, gpw_cultiv_grass):

    # Create default token array and default node code array from LC timeseries
    tokens = [token_for_lc(v) for v in lc_timeseries]               #char array representing land use timeseries
    node_codes = [default_node_code(token) for token in tokens]     #int array representing class definition rules applied throughout the timeseries

    lu_dict = {
        "initial_tokens": tokens.copy(),
        "tokens": tokens,
        "node_codes": node_codes,
        "driver": driver,
        "tcl_year": tcl_year,
        "tcl_prior": (tcl_year != 0 and tcl_year <= min(years_annual)),  # convert to bool #TODO: Don't precalculate?
        "pre_2000_plantation": (pre_2000_plantation == 1),
        "planting_year": planting_year,
        "sdpt_oil_palm": (sdpt_oil_palm == 1),
        "sdpt_tree_crop": sdpt_tree_crop,
        "sdpt_planted_forest": sdpt_planted_forest,
        "gmw_mangrove": gmw_mangrove,
        "gpw_cultiv_grass": gpw_cultiv_grass,
    }

    # Check if oil palm, tree crop or forest based on special cases
    extent_rule_applied = apply_extent_rules(lu_dict)

    if not extent_rule_applied:
        token_seq = "".join(lu_dict["tokens"]) #Creates a concat string
        if "S" in token_seq and not re.fullmatch(r"S+", token_seq):
            apply_built_transition(lu_dict)
        elif "C" in token_seq and not re.fullmatch(r"C+", token_seq):
            apply_crop_transition(lu_dict)
        elif re.fullmatch(r"F+", token_seq):
            apply_all_tall_veg(lu_dict)
        elif re.fullmatch(r"G+", token_seq):
            apply_all_short_veg(lu_dict)
        elif re.fullmatch(r"[FG]+", token_seq):
            apply_tall_short(lu_dict)
        elif re.fullmatch(r"[GB]+", token_seq):
            apply_short_bare(lu_dict)
        elif re.fullmatch(r"[FGBWO]+", token_seq) and re.search(r"[FGB]", token_seq) and re.search(r"[WO]", token_seq):
            apply_veg_bare_water(lu_dict)
        elif re.fullmatch(r"[WO]+", token_seq):
            apply_wetland_water(lu_dict)


    # Final token and node code timeseries
    final_tokens = lu_dict["tokens"]
    node_code_ts = lu_dict["node_codes"]

    # Convert final tokens to numeric LU codes
    lu_ts = [lu_token_map[token] for token in final_tokens]

    # Check that there is only one land use transition during the timeseries
    check_single_lu_transition(lu_dict, lu_ts)

    # Create transition timeseries: 2015_2016 through 2023_2024
    transition_ts = [int(f"{lu_ts[i]}{lu_ts[i + 1]}") for i in range(len(lu_ts) - 1)]

    # Create sequential unique LU summary ([3, 3, 3, 4, 4, 4, 2, 2, 2] -> [3, 4, 2] -> Forest to Grass to Crop)
    summary = []
    for lu in lu_ts:
        if not summary or lu != summary[-1]:
            summary.append(lu)

    return lu_ts, node_code_ts, transition_ts, summary


##### Functions to run IPCC land use classification on tabular data

In [ ]:
# Dictionaries to convert numeric output to text for export csv
driver_code_map = {
    1: "permanent_agriculture",
    2: "hard_commodities",
    3: "shifting_cultivation",
    4: "logging",
    5: "wildfire",
    6: "settlements_infrastructure",
    7: "other_natural_disturbances"
}

lu_code_map = {
    1: "settlements_infrastructure",
    2: "cropland",
    3: "forest",
    4: "grassland",
    5: "wetland",
    6: "other",
}

# Reverse lookup because node_code_map is name -> code, but export needs code -> name.
node_code_text_map = {v: k for k, v in node_code_map.items()}

# Converts input csv values as boolean values for classification rules
def as_bool(v):
    if pd.isna(v):
        return False
    if isinstance(v, str):
        v = v.strip().lower()
        if v in {"", "false", "f", "no", "n", "0"}:
            return False
        if v in {"true", "t", "yes", "y", "1"}:
            return True
        return False
    return bool(v)

# Converts input csv values as int values for classification rules
def as_int_or_0(v):
    if pd.isna(v):
        return int(0)
    try:
        return int(v)
    except (TypeError, ValueError):
        return int(0)

# Converts input csv values as float values for classification rules
def as_float_or_0(v):
    if pd.isna(v):
        return 0.0
    try:
        return float(v)
    except (TypeError, ValueError):
        return 0.0

# Iterate through each scenario and classify LC to create LU timeseries
def classify_dataframe(df):
    out = []
    lc_cols = [f"lc_{y}" for y in years_annual]
    for idx, row in df.iterrows():
        scenario_id = row.get("id", idx)
        lc_ts=[as_int_or_0(row.get(lc)) for lc in lc_cols]        #Array of ints
        driver=as_int_or_0(row.get("driver"))                               #Int
        tcl_year=as_int_or_0(row.get("tcl_year"))                           #Int
        pre_2000_plantation = as_int_or_0(row.get("pre_2000_plantation"))   #Int
        planting_year = as_float_or_0(row.get("planting_year"))               #Float
        sdpt_oil_palm = as_int_or_0(row.get("sdpt_oil_palm"))               #Int
        sdpt_tree_crop=as_bool(row.get("sdpt_tree_crop"))               #Boolean
        sdpt_planted_forest=as_bool(row.get("sdpt_planted_forest"))     #Boolean
        gmw_mangrove=as_bool(row.get("gmw_mangrove"))                   #Boolean
        gpw_cultiv_grass=as_bool(row.get("gpw_cultiv_grass"))           #Boolean

        lu_ts, node_code_ts, transition_ts, summary = apply_regex_rules(lc_ts, driver, tcl_year, pre_2000_plantation, planting_year, sdpt_oil_palm, sdpt_tree_crop, sdpt_planted_forest, gmw_mangrove, gpw_cultiv_grass)

        # Convert numeric values to text values in export columns
        # Land use timeseries
        lu_cols = {f"LU_{y}": lu_code_map.get(lu_ts[i], "unknown") for i, y in enumerate(years_annual)}

        # Node code timeseries
        node_code_cols = {f"node_{y}": node_code_text_map.get(node_code_ts[i], "unknown") for i, y in enumerate(years_annual)}

        # Land use transition timeseries (create column to flag whether a transition happened at all)
        conversion = False  # Land use transition flag
        trans_cols = {}

        for i, (a, b) in enumerate(zip(years_annual[:-1], years_annual[1:])):
            transition_code = int(transition_ts[i])

            from_code = transition_code // 10
            to_code = transition_code % 10

            from_lu = lu_code_map[from_code]
            to_lu = lu_code_map[to_code]

            key = f"LU_{a}_{b}"

            if from_code == to_code:
                trans_cols[key] = f"{from_lu} remaining {to_lu}"
            else:
                trans_cols[key] = f"{from_lu} to {to_lu}"
                conversion = True

        # Summary all LU transitions that occurred during entire timeseries
        summary_text = " to ".join(lu_code_map.get(lu, str(lu))for lu in summary)

        out.append({
            "id": scenario_id,
            "tcl_year": (np.nan if tcl_year == 0 else tcl_year),
            "driver": (np.nan if driver == 0 else driver_code_map.get(driver, str(driver))),
            "pre_2000_plantation": True if pre_2000_plantation else np.nan,
            "planting_year": np.nan if planting_year == 0 else planting_year,
            "sdpt_oil_palm": True if sdpt_oil_palm else np.nan,
            "sdpt_tree_crop": True if sdpt_tree_crop else np.nan,
            "sdpt_planted_forest": True if sdpt_planted_forest else np.nan,
            "gmw_mangrove": True if gmw_mangrove else np.nan,
            "gpw_cultiv_grass": True if gpw_cultiv_grass else np.nan,
            **lu_cols,
            **node_code_cols,
            **trans_cols,
            "summary": summary_text,
            "conversion_occurred": conversion,
        })

    return pd.DataFrame(out)

##### Read in scenarios from spreadsheet and export the resulting land use classification spreadsheet

In [ ]:
# Read in data
file_path = "/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/LUC/conversion_LUC_scenarios.xlsx"
sheet_name = "scenarios"
scenarios_df = pd.read_excel(file_path, sheet_name=sheet_name)

# Coerce to numeric
lc_cols = [f"lc_{y}" for y in years_annual]
numeric_cols = lc_cols + ["driver", "tcl_year"]
for c in numeric_cols:
    if c in scenarios_df.columns:
        scenarios_df[c] = pd.to_numeric(scenarios_df[c], errors="coerce")

# Run land use classification
results_df = classify_dataframe(scenarios_df)

# Export results to xlsx
results_df.to_excel("/mnt/c/GIS/git/AFOLU_GHG_flux_model/src/LULUCF/scripts/postprocessing/LUC/conversion_LUC_scenario_results.xlsx", index=False)

# Print results
print("\nClassification Results:")
print(results_df)